In [ ]:
!pip install -qU uv
!uv pip install -r ../requirements.txt

In [42]:
import os
from openai import OpenAI
from dotenv import dotenv_values

config = dotenv_values("../.env") 
client = OpenAI(
    api_key=config.get("OPENAI_API_KEY"),
)

## Build unified dataframes

In [34]:
import pandas as pd
import xml.etree.ElementTree as ET
import re

# make pandas .head print full data
pd.set_option('display.max_colwidth', None)

subtask1A_input_path = "../../datasets/TaskA_Input.xml"
subtask1A_labels_path = "../../datasets/TaskA_Input.xml"

subtask1B_input_path = "../../datasets/TaskB_Input.xml"
subtask1B_labels_path = "../../datasets/TaskB_GT.tsv"

subtask1C_input_path = "../../datasets/TaskC_Input.xml"
subtask1C_labels_path = "../../datasets/TaskC_GT.tsv"

In [35]:
def parse_xml_to_dataframe(xml_path):
    """Parse XML file and return DataFrame with Question_ID and Response text"""
    
    # Read the file with UTF-8 encoding
    with open(xml_path, 'r', encoding='utf-8') as file:
        content = file.read()
    
    # Split content by <Question> tags to get individual questions
    question_blocks = re.split(r'<Question>', content)
    question_blocks = [block.strip() for block in question_blocks if block.strip()]
    
    data = []
    for i, block in enumerate(question_blocks):
        try:
            # Add the opening tag back and ensure closing tag
            if not block.endswith('</Question>'):
                # Find the last </Question> or add one if missing
                last_question_end = block.rfind('</Question>')
                if last_question_end == -1:
                    block += '</Question>'
                else:
                    # Keep only content up to the last </Question>
                    block = block[:last_question_end + len('</Question>')]
            
            # Wrap in Question tags
            question_xml = f'<Question>{block}'
            
            # Parse this individual question
            try:
                question_element = ET.fromstring(question_xml)
            except ET.ParseError:
                # If parsing fails, try to extract data manually using regex
                question_id = re.search(r'<ID>(.*?)</ID>', question_xml, re.DOTALL)
                model = re.search(r'<Model>(.*?)</Model>', question_xml, re.DOTALL)
                text = re.search(r'<Text>(.*?)</Text>', question_xml, re.DOTALL)
                response = re.search(r'<Response>(.*?)</Response>', question_xml, re.DOTALL)
                
                data.append({
                    'Question_ID': question_id.group(1).strip() if question_id else f'Q{i+1:03d}',
                    'Model': model.group(1).strip() if model else 'Unknown',
                    'Text': text.group(1).strip() if text else '',
                    'Response': response.group(1).strip() if response else ''
                })
                continue
            
            # Extract data from parsed element
            question_id_elem = question_element.find('ID')
            model_elem = question_element.find('Model')
            text_elem = question_element.find('Text')
            response_elem = question_element.find('Response')
            
            question_id = question_id_elem.text if question_id_elem is not None else f'Q{i+1:03d}'
            model = model_elem.text if model_elem is not None else 'Unknown'
            text = text_elem.text if text_elem is not None else ''
            response = response_elem.text if response_elem is not None else ''
            
            data.append({
                'Question_ID': question_id,
                'Model': model,
                'Text': text,
                'Response': response
            })
            
        except Exception as e:
            print(f"Error processing question block {i+1}: {e}")
            continue
    
    return pd.DataFrame(data)

# Parse all input XML files
print("Parsing XML files...")
df_a_input = parse_xml_to_dataframe(subtask1A_input_path)
df_b_input = parse_xml_to_dataframe(subtask1B_input_path)
df_c_input = parse_xml_to_dataframe(subtask1C_input_path)

# Read ground truth TSV files
df_a_gt = pd.read_csv("../../datasets/TaskA_GT.tsv", sep='\t')
df_b_gt = pd.read_csv(subtask1B_labels_path, sep='\t')
df_c_gt = pd.read_csv(subtask1C_labels_path, sep='\t')

print("Input data shapes:")
print(f"Task A Input: {df_a_input.shape}")
print(f"Task B Input: {df_b_input.shape}")
print(f"Task C Input: {df_c_input.shape}")

print("\nGround truth data shapes:")
print(f"Task A GT: {df_a_gt.shape}")
print(f"Task B GT: {df_b_gt.shape}") 
print(f"Task C GT: {df_c_gt.shape}")

# Show sample data
print("\nSample from Task A Input:")
print(df_a_input.head(2))

Parsing XML files...
Input data shapes:
Task A Input: (50, 4)
Task B Input: (50, 4)
Task C Input: (50, 4)

Ground truth data shapes:
Task A GT: (210, 6)
Task B GT: (247, 6)
Task C GT: (179, 6)

Sample from Task A Input:
  Question_ID    Model  \
0       A-Q01  Model-5   
1       A-Q02  Model-1   

                                                                                                                              Text  \
0                                                                                              من هن المحرمات من النساء في الزواج؟   
1  لماذا تزوج النبي (ص) أكثر من أربعة نساء مع ان الآية 3  من سورة النساء ورد فيها "فانكحوا ما طاب لكم من النساء مثنى وثلاث ورباع"؟   

                                                                                                                                                                                                                                                                                                          

In [36]:
# Create unified datasets by merging input and ground truth data

# Task A: Merge input with ground truth labels
df_a_unified = df_a_input.merge(df_a_gt, on='Question_ID', how='left')

# Task B: Merge input with ground truth labels  
df_b_unified = df_b_input.merge(df_b_gt, on='Question_ID', how='left')

# Task C: Merge input with ground truth labels
df_c_unified = df_c_input.merge(df_c_gt, on='Question_ID', how='left')

# Save unified datasets as TSV files
df_a_unified.to_csv("../datasets/taskA_unified.tsv", sep='\t', index=False)
df_b_unified.to_csv("../datasets/taskB_unified.tsv", sep='\t', index=False)
df_c_unified.to_csv("../datasets/taskC_unified.tsv", sep='\t', index=False)

print("Unified datasets created:")
print(f"Task A unified: {df_a_unified.shape}")
print(f"Task B unified: {df_b_unified.shape}")
print(f"Task C unified: {df_c_unified.shape}")


Unified datasets created:
Task A unified: (210, 9)
Task B unified: (247, 9)
Task C unified: (179, 9)


In [24]:
df_b_unified.head(1)

,Question_ID,Model,Text,Response,Annotation_ID,Label,Span_Start,Span_End,Original_Span
0,B-Q01,Model-6,ما معنى صلى الله عليه وسلم؟,"\nالسؤال: ما معنى ""صلى الله عليه وسلم""؟\n**الإجابة:**\nعبارة **""صلى الله عليه وسلم""** هي توجيهٌ عظيم من الله تعالى لرفع سُبحانه وجلالةٍ عن النبي محمد ﷺ، وتشمل طلبًا للدعاء عليه من الله، وطلبًا للسلام عليه، وتأكيدًا على نبوته وعِبادته. تُستخدم هذه العبارة في التحية، والدعاء، والذكر، وتقدير سُنَّة النبي ﷺ، وتحقيقًا لقول الله تعالى:\n> **""إِنَّ اللَّهَ وَمَلَائِكَتَهُ يُحِبُّونَ الْمُحْسِنِينَ""** \n> **[الحج: 77]** \n> أي أن الله وملائكته يحبون من يُحسنون الفعل، ومنهم النبي محمد ﷺ الذي هو ""المرسي"" (المرسى عليه السلام)، وهو أفضل خلق الله.\n### **الإعجاز القرآني:**\n1. **السورة 33 (الأحزاب): 56** \n > **""إِنَّ اللَّهَ يُحِبُّ التَّوْبَةَ وَيُحِبُّ الْمُحْسِنِينَ""** \n ويدلّ على أن النبي ﷺ من المُحسنين، ويتّصف بالمحبوبية من الله.\n2. **السورة 2 (البقرة): 255** \n > **""إِنَّكَ لَتَعْبُدُ اللَّهَ إِنْ كُنْتَ صَادِقًا""** \n يُشير إلى أن النبي ﷺ هو ""المرسي"" (المرسي عليه السلام) الذي يُعَبَّد، وهو مَن يُحَمَّدُ ويُسَمَّى بِالصَّلاةِ والسلام.\n### **الإعجاز النبوي:**\n1. **حديث صحيح",1,WrongAyah,334,389,إِنَّ اللَ...ُحْسِنِينَ


In [37]:
# validae Original_Span detection

ix = 0
print(ix, "->" ,df_a_unified.iloc[ix]["Response"][ df_a_unified.iloc[ix]["Span_Start"]: df_a_unified.iloc[ix]["Span_End"] ])

ix = 12
print(ix, "->" ,df_b_unified.iloc[ix]["Response"][ df_b_unified.iloc[ix]["Span_Start"]: df_b_unified.iloc[ix]["Span_End"] ])

0 -> لا تنكح المرأة على عمتها ولا على خالتها
12 -> وَلِكُلٍّ جَعَلْنَا مَوَالِيَ مِمَّا تَرَكَ الْوَالِدَانِ وَالأَقْرَبُونَ


In [40]:
# iterate over all rows and replace Original_Span with the actual span from Response
# replce all Original_Span values with `df_a_unified.iloc[ix]["Response"][ df_a_unified.iloc[ix]["Span_Start"]: df_a_unified.iloc[ix]["Span_End"] ]`

for _df in [df_a_unified, df_b_unified, df_c_unified]:
    for ix in range(len(_df)):
        _df.at[ix, "Original_Span"] = _df.iloc[ix]["Response"][ _df.iloc[ix]["Span_Start"]: _df.iloc[ix]["Span_End"] ]

# save updated unified datasets
df_a_unified.to_csv("../datasets/taskA_unified.tsv", sep='\t', index=False)
df_b_unified.to_csv("../datasets/taskB_unified.tsv", sep='\t', index=False)
df_c_unified.to_csv("../datasets/taskC_unified.tsv", sep='\t', index=False)

In [41]:
for _df, _name in zip([df_a_unified, df_b_unified, df_c_unified], ['Task A', 'Task B', 'Task C']):
    print("Task", _name)
    print(_df['Label'].value_counts())
    print("-"*50)

Task Task A
Label
Ayah            118
Hadith           76
NoAnnotation     16
Name: count, dtype: int64
--------------------------------------------------
Task Task B
Label
CorrectAyah      110
WrongAyah         70
CorrectHadith     37
WrongHadith       30
Name: count, dtype: int64
--------------------------------------------------
Task Task C
Label
WrongAyah      95
WrongHadith    84
Name: count, dtype: int64
--------------------------------------------------


## Load the unified dataset

In [43]:
import pandas as pd

# Load the unified dataset
taskA_df = pd.read_csv("../datasets/taskA_unified.tsv", sep='\t')
taskB_df = pd.read_csv("../datasets/taskB_unified.tsv", sep='\t')
taskC_df = pd.read_csv("../datasets/taskC_unified.tsv", sep='\t')

In [45]:
taskA_df.sample(frac=1).iloc[:10].to_csv("../datasets/taskA_unified_sample.tsv", sep='\t', index=False)

In [48]:
input_text = """
نعم، يمكن أن يكون الابتلاء بفتح الرزق والخيرات عمومًا، وهو من أنواع الابتلاء التي اختبر بها الله تعالى عباده لاختبار صبرهم وشكرهم. يُذكر في القرآن الكريم والحديث النبوي أن الابتلاء قد يشمل الظروف التي تُختبر فيها صدق الإيمان وحسن التصرف في مسائل الرزق والخيرات، سواء بفتحها أو بمنعها. إليك بعض الآيات والأحاديث التي تدعم هذا المفهوم:
### **آيات من القرآن الكريم:**
1. **سورة الأنكابوت (29:2-3):** 
 > {وَإِنَّكَ لَعَلَىٰ خُلُقٍ عَظِيمٍ * لَّأَمْلَأَنَّ جَهَنَّمَ صُدُورَ أَصْدِقَاءَكَ الَّذِينَ كَفَرُوا بِآيَاتِ اللَّهِ وَأَخَذَتْهُمْ فِي الْعَذَابِ مَفْزَزَةً * أَوْ لَمْ يَكُونُوا مُؤْمِنِينَ} 
 - في هذه الآية، يُذكر أن الله يختبر الناس بـ **الخيرات** (النفوس الصادقة) والرزق، حيث يُعدّ من أسباب الابتلاء أن يُختبر الناس بصدق إيمانهم وطاعتهم، سواء بمنحهم الخيرات أو بمنعها. 
2. **سورة الكهف (18:64):** 
 > {وَإِنَّكَ لَعَلَىٰ خُلُقٍ عَظِيمٍ * لَّأَمْلَأَنَّ جَهَنَّمَ صُدُورَ أَصْدِقَاءَكَ الَّذِينَ كَفَرُوا بِآيَاتِ اللَّهِ وَأَخَذَتْهُمْ فِي الْعَذَابِ مَفْزَزَ""".strip()

search_text = """9:2-3):** 
 > {وَإِنَّكَ لَعَلَىٰ خُلُقٍ عَظِيمٍ * لَّأَمْلَأَنَّ جَهَنَّمَ صُدُورَ أَصْدِقَاءَكَ الَّذِينَ كَفَرُوا بِآيَاتِ اللَّهِ وَأَخَذَتْهُمْ فِي الْعَذَابِ مَفْزَزَةً * أَوْ لَمْ يَكُونُوا مُؤْ"""

print("Span Start:", input_text.index(search_text))
print("Span End:", input_text.index(search_text) + len(search_text))

Span Start: 387
Span End: 588


In [60]:
import pandas as pd

submission_a_df = pd.read_csv("./tasks_output/test_Subtask_1A_submission-v2.tsv", sep='\t')
submission_a_df.head(2)

,Question_ID,Span_Start,Span_End,Span_Type,Original_Span,LLM_Value
0,A-Q001,403,595,Ayah,وَاِنَّكَ لَعَلَيٰ خُلُقٍ عَظِيمٍ * لَّاَمْلَاَنَّ جَهَنَّمَ صُدُورَ اَصْدِقَاءَكَ الَّذِينَ كَفَرُوا بِايَاتِ اللَّهِ وَاَخَذَتْهُمْ فِي الْعَذَابِ مَفْزَزَةً * اَوْ لَمْ يَكُونُوا مُوْمِنِينَ,وَاِنَّكَ لَعَلَيٰ خُلُقٍ عَظِيمٍ * لَّاَمْلَاَنَّ جَهَنَّمَ صُدُورَ اَصْدِقَاءَكَ الَّذِينَ كَفَرُوا بِايَاتِ اللَّهِ وَاَخَذَتْهُمْ فِي الْعَذَابِ مَفْزَزَةً * اَوْ لَمْ يَكُونُوا مُوْمِنِينَ
1,A-Q001,812,968,Ayah,وَاِنَّكَ لَعَلَيٰ خُلُقٍ عَظِيمٍ * لَّاَمْلَاَنَّ جَهَنَّمَ صُدُورَ اَصْدِقَاءَكَ الَّذِينَ كَفَرُوا بِايَاتِ اللَّهِ وَاَخَذَتْهُمْ فِي الْعَذَابِ مَفْزَزَ,وَاِنَّكَ لَعَلَيٰ خُلُقٍ عَظِيمٍ * لَّاَمْلَاَنَّ جَهَنَّمَ صُدُورَ اَصْدِقَاءَكَ الَّذِينَ كَفَرُوا بِايَاتِ اللَّهِ وَاَخَذَتْهُمْ فِي الْعَذَابِ مَفْزَزَ


In [61]:
# remove LLM_Value and Original_Span and save the tsv again with `-noverbose.tsv`
submission_a_df_noverbose = submission_a_df.drop(columns=['LLM_Value', 'Original_Span'])
submission_a_df_noverbose.to_csv("./tasks_output/test_Subtask_1A_submission-v2-noverbose.tsv", sep='\t', index=False)

In [55]:
import pandas as pd

submission_b_df = pd.read_csv("./tasks_output/submission_task_B_verbose.tsv", sep='\t')
submission_b_df.head(2)

,Sequence_ID,Label,Original_Span,Span_Type,Top_Found_Results
0,1,Correct,"وَلَنَبْلُوَنَّكُمْ بِشَيْءٍ مِنَ الْخَوْفِ وَالْجُوعِ وَنَقْصٍ مِنَ الْأَمْوَالِ وَالْأَنْفُسِ وَالثَّمَرَاتِ وَبَشِّرِ الصَّابِرِينَ""",Ayah,وَلَنَبْلُوَنَّكُمْ بِشَيْءٍ مِنَ الْخَوْفِ وَالْجُوعِ وَنَقْصٍ مِنَ الْأَمْوَالِ وَالْأَنْفُسِ وَالثَّمَرَاتِ ۗ وَبَشِّرِ الصَّابِرِينَ
1,2,Correct,"ما يصيب المؤمن من شوكة فما فوقها إلا رفعه الله بها درجة، أو حط عنه بها خطيئة""",Hadith,مَا يُصِيبُ الْمُؤْمِنَ مِنْ شَوْكَةٍ فَمَا فَوْقَهَا إِلَّا رَفَعَهُ اللهُ بِهَا دَرَجَةً ، أَوْ حَطَّ عَنْهُ بِهَا خَطِيئَةً .


In [56]:
# drop Original_Span Span_Type Top_Found_Results
submission_b_df_noverbose = submission_b_df.drop(columns=['Original_Span', 'Span_Type', 'Top_Found_Results'])
submission_b_df_noverbose.to_csv("./tasks_output/submission_task_B_noverbose.tsv", sep='\t', index=False)

In [92]:
import pandas as pd

submission_c_df = pd.read_csv("./tasks_output/submission_task_C_v4.tsv", sep='\t')
submission_c_df.head(2)

Sequence_ID  \
0            1   
1            2   

                                                                                                                                                                                      Correction  \
0                                                                                                                                                                                            خطأ   
1  إِنَّ الَّذِينَ تَوَلَّوْا مِنْكُمْ يَوْمَ الْتَقَى الْجَمْعَانِ إِنَّمَا اسْتَزَلَّهُمُ الشَّيْطَانُ بِبَعْضِ مَا كَسَبُوا ۖ وَلَقَدْ عَفَا اللَّهُ عَنْهُمْ ۗ إِنَّ اللَّهَ غَفُورٌ حَلِيمٌ   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                Original_Span  \
0  وَالَّذِينَ يَرْمُونَ الْمُحْصَنَاتِ ثُمَّ لَمْ يَأْتُوا بِأَرْبَعَةٍ شُهُودٍ فَاقِضْ لَهُنَّ نِصْفَ مَا زَعَمْنَ إِنْ أَرْبَعَةٌ فَمِنْ أَهْلِهِنَّ فَلْيُؤْتِهِنَّ نِصْفَ مَا تَمَل्लَكْنَ يَومَ عِرْضِهِنَّ وَيَحْلِفْنَ إِنْ أَرْبَعَةٌ فَإِنْ لَمْ يَأْتُوا بِأَرْبَعَةٍ فَآخَرُ أَنْ تُضَامَنَ نِسَاءٌ بِغَيْرِ ذَلِكَ وَلَوْ كَانَ أَحَدُهُنَّ مِنْ أَزْوَاجِ أَزْوَاجِكُمْ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُمْ مِنْ عَدْلٍ وَلَوْ كَانَ أَحَدُهُنَّ أَخَوَةٌ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُمْ مِنْ رَحِمٍ وَلَيْسَ لَكُمْ أَنْ تَكُونُوا عَلَى الْقَوْلِ أَكُلٌ عَلَى مِثْلِ مَا عَمِلَتْ بِهِ نِسَاءٌ مِنْ قَبْلُ عَلَى أَنْ تَصَدَقْنَ خَيْرًا بِالْمَعْرُوفِ وَلَوْ كَانَ أَحَدُهُنَّ مَرْضِيَةً مِنْكُمْ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُمْ مِنْ عَدْلٍ وَلَوْ كَانَ أَحَدُهُنَّ أُخْتُكُمْ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُمْ مِنْ رَحِمٍ وَلَيْسَ لَكُمْ أَنْ تَكُونُوا عَلَى الْقَوْلِ أَكُلٌ عَلَى مِثْلِ مَا عَمِلَتْ بِهِ نِسَاءٌ مِنْ قَبْلُ عَلَى أَنْ تَصَدَقْنَ خَيْرًا بِالْمَعْرُوفِ وَلَيْسَ لَكُمْ أَنْ تَخُضَنَ إِلى مَعْشَرٍ بَعْدَهُنَّ مِنْ بَعْدٍ وَلَيْسَ لَكُمْ أَنْ تَرْجِعَ إِلَى عُقُولِ مُحْصِنَاتٍ بَعْدُ وَلَوْ كَانَ أَحَدُهُنَّ أُمّاً لَكُمْ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُمْ مِنْ عَدْلٍ وَلَوْ كَانَ أَحَدُهُنَّ أَخَوَةً لَكُمْ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُمْ مِنْ رَحِمٍ وَلَيْسَ لَكُمْ أَنْ تَكُونُوا عَلَى الْقَوْلِ أَكُلٌ عَلَى مِثْلِ مَا عَمِلَتْ بِهِ نِسَاءٌ مِنْ قَبْلُ عَلَى أَنْ تَصَدَقْنَ خَيْرًا بِالْمَعْرُوفِ وَلَوْ كَانَ أَحَدُهُنَّ بَيتُكُمْ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُمْ مِنْ عَدْلٍ وَلَوْ كَانَ أَحَدُهُنَّ مِنْ أَزْوَاجِكُمْ فَلَمْ تَكُنْ لَهُنَّ أَحَدٌ مِنْ دُونِكُ

In [93]:
# drop Original_Span Span_Type Top_Found_Results
submission_c_df_noverbose = submission_c_df.drop(columns=['Original_Span', 'Span_Type', 'Top_Found_Results'])
submission_c_df_noverbose.to_csv("./tasks_output/submission_task_C_v4_noverbose.tsv", sep='\t', index=False)

## Validate Cohere

In [ ]:
!pip install cohere

In [ ]:
import cohere
from numpy import dot
from numpy.linalg import norm

from dotenv import dotenv_values

config = dotenv_values("../.env") 

co = cohere.ClientV2(api_key=config.get("COHERE_API_KEY"))

text_inputs = [
    {
        "content": [
            {"type": "text", "text": "لا يكمل إيمان أحدكم حتى يحب لأخيه ما يحبه لنفسه"},
        ]
    },
    {
        "content": [
            {"type": "text", "text": "وَلَقَدْ أَرْسَلْنَا مِنْ قَبْلِكَ فِي شِيَعِ الْأَوَّلِينَ"},
        ]
    },
]

response = co.embed(
    inputs=text_inputs,
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
)

a = response.embeddings.float[0]
b = response.embeddings.float[1]

cos_sim = dot(a, b)/(norm(a)*norm(b))
print("Cosine Similarity:", cos_sim)

Cosine Similarity: 0.15367912698854017


In [85]:
len(a)

1536